In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


In [2]:
def load_mnist():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    x_train = train_set.data.view(-1, 784).float() / 255.0
    y_train = train_set.targets
    x_test = test_set.data.view(-1, 784).float() / 255.0
    y_test = test_set.targets

    return x_train.to(device), y_train.to(device), x_test.to(device), y_test.to(device)

In [3]:
def one_nn_classify(prototypes, proto_labels, queries):
    dists = torch.cdist(queries, prototypes)
    nn_indices = torch.argmin(dists, dim=1)
    return proto_labels[nn_indices]

In [ ]:
class LVQ:
    def __init__(self, M, lr=0.01, epochs=5, adaptive_lr=True):
        self.M = M
        self.lr = lr
        self.epochs = epochs
        self.adaptive_lr = adaptive_lr

    def fit(self, x_train, y_train, selection=False):
        N, D = x_train.shape
        if selection:
            indices = torch.randperm(N)[:self.M]
            self.prototypes = x_train[indices].clone()
            self.proto_labels = y_train[indices].clone()
        else:
            indices = torch.randperm(N)[:self.M]
            self.prototypes = x_train[indices].clone()
            self.proto_labels = y_train[indices].clone()

        lr = self.lr
        for epoch in range(self.epochs):
            perm = torch.randperm(N)
            x_epoch = x_train[perm]
            y_epoch = y_train[perm]
            # batch update
            for i in range(N):
                x = x_epoch[i]
                y = y_epoch[i]
                dists = torch.norm(self.prototypes - x, dim=1)
                idx = torch.argmin(dists)
                if self.proto_labels[idx] == y:
                    self.prototypes[idx] += lr * (x - self.prototypes[idx])
                else:
                    self.prototypes[idx] -= lr * (x - self.prototypes[idx])
            if self.adaptive_lr:
                lr *= 0.9
        return self.prototypes, self.proto_labels

In [ ]:
class LVQ3(LVQ):
    def __init__(self, M, lr=0.01, epochs=5, epsilon=0.3, adaptive_lr=True):
        super().__init__(M, lr, epochs, adaptive_lr)
        self.epsilon = epsilon

    def fit(self, x_train, y_train, selection=False):
        N, D = x_train.shape
        if selection:
            indices = torch.randperm(N)[:self.M]
            self.prototypes = x_train[indices].clone()
            self.proto_labels = y_train[indices].clone()
        else:
            indices = torch.randperm(N)[:self.M]
            self.prototypes = x_train[indices].clone()
            self.proto_labels = y_train[indices].clone()

        lr = self.lr
        for epoch in range(self.epochs):
            perm = torch.randperm(N)
            x_epoch = x_train[perm]
            y_epoch = y_train[perm]
            for i in range(N):
                x = x_epoch[i]
                y = y_epoch[i]
                dists = torch.norm(self.prototypes - x, dim=1)
                sorted_idx = torch.argsort(dists)[:2]
                p1, p2 = sorted_idx[0], sorted_idx[1]

                if self.proto_labels[p1] == y:
                    self.prototypes[p1] += lr * (x - self.prototypes[p1])
                else:
                    self.prototypes[p1] -= lr * (x - self.prototypes[p1])

                if self.proto_labels[p2] == y:
                    self.prototypes[p2] += self.epsilon * lr * (x - self.prototypes[p2])
                else:
                    self.prototypes[p2] -= self.epsilon * lr * (x - self.prototypes[p2])

            if self.adaptive_lr:
                lr *= 0.9
        return self.prototypes, self.proto_labels

In [6]:
def random_selection(x_train, y_train, M):
    indices = torch.randperm(len(x_train))[:M]
    return x_train[indices].clone(), y_train[indices].clone()

In [7]:
def run_experiment(M_list=[1000, 5000, 10000], repeats=5):
    x_train, y_train, x_test, y_test = load_mnist()
    results = {}

    for M in M_list:
        results[M] = {}
        print(f"\n--- M = {M} ---")

        # Random selection baseline
        acc_rand_list = []
        for r in tqdm(range(repeats), desc='Random'):
            p_rand, l_rand = random_selection(x_train, y_train, M)
            preds_rand = one_nn_classify(p_rand, l_rand, x_test)
            acc_rand_list.append((preds_rand == y_test).float().mean().item())
        results[M]['Random'] = acc_rand_list

        # LVQ prototype generation
        acc_lvq_gen_list = []
        for r in tqdm(range(repeats), desc='LVQ-Gen'):
            lvq = LVQ(M, lr=0.01, epochs=5, adaptive_lr=True)
            p, l = lvq.fit(x_train, y_train, selection=False)
            preds = one_nn_classify(p, l, x_test)
            acc_lvq_gen_list.append((preds == y_test).float().mean().item())
        results[M]['LVQ-Gen'] = acc_lvq_gen_list

        # LVQ prototype selection
        acc_lvq_sel_list = []
        for r in tqdm(range(repeats), desc='LVQ-Sel'):
            lvq = LVQ(M, lr=0.01, epochs=5, adaptive_lr=True)
            p, l = lvq.fit(x_train, y_train, selection=True)
            preds = one_nn_classify(p, l, x_test)
            acc_lvq_sel_list.append((preds == y_test).float().mean().item())
        results[M]['LVQ-Sel'] = acc_lvq_sel_list

        # LVQ3 prototype generation
        acc_lvq3_gen_list = []
        for r in tqdm(range(repeats), desc='LVQ3-Gen'):
            lvq3 = LVQ3(M, lr=0.01, epochs=5, epsilon=0.3, adaptive_lr=True)
            p, l = lvq3.fit(x_train, y_train, selection=False)
            preds = one_nn_classify(p, l, x_test)
            acc_lvq3_gen_list.append((preds == y_test).float().mean().item())
        results[M]['LVQ3-Gen'] = acc_lvq3_gen_list

        # LVQ3 prototype selection
        acc_lvq3_sel_list = []
        for r in tqdm(range(repeats), desc='LVQ3-Sel'):
            lvq3 = LVQ3(M, lr=0.01, epochs=5, epsilon=0.3, adaptive_lr=True)
            p, l = lvq3.fit(x_train, y_train, selection=True)
            preds = one_nn_classify(p, l, x_test)
            acc_lvq3_sel_list.append((preds == y_test).float().mean().item())
        results[M]['LVQ3-Sel'] = acc_lvq3_sel_list

        # Print results
        print(f"Results for M={M}:")
        for method_name, acc_list in results[M].items():
            print(f"  {method_name}: {acc_list}")
            if repeats > 1:
                print(f"    Mean: {np.mean(acc_list):.4f}")

    return results

In [ ]:
results = run_experiment(M_list=[500, 1000, 2000, 5000, 10000], repeats=1)


--- M = 1000 ---


LVQ3-Sel: 100%|██████████| 1/1 [01:59<00:00, 119.28s/it]


Results for M=1000:
  Random: [0.8837999701499939]
  LVQ-Gen: [0.9458999633789062]
  LVQ-Sel: [0.948699951171875]
  LVQ3-Gen: [0.9511999487876892]
  LVQ3-Sel: [0.9497999548912048]

--- M = 5000 ---


LVQ3-Sel:   0%|          | 0/1 [00:00<?, ?it/s]